# Carousel Metric Analysis

This notebook is a self-contained, notebook-first reorganization of the original `carousel_metric.ipynb`.

It does not import any project-defined `carousel_metric` package functions. All data cleaning, plotting, discount functions, metrics, and simulation helpers are defined directly inside this notebook.

The raw original notebook is kept at `notebooks/carousel_metric_original.ipynb` for reference.

## 0. Setup

Run this notebook from either the repository root or the `notebooks/` directory.

Expected input files:

- `data/raw/summary_feedback.csv`
- `data/raw/click_summary_dataset.csv`

Generated figures, tables, and reports are written to `outputs/`.

In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import linear_sum_assignment
from scipy.stats import pearsonr, spearmanr
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INTERACTIONS_CSV = DATA_DIR / "summary_feedback.csv"
CLICKS_CSV = DATA_DIR / "click_summary_dataset.csv"

TARGET_GROUP = "uva"      # one of: "overall", "kinit", "uva"
N_TRIALS = 20_000
RNG_SEED = 42

sns.set_theme(style="white", context="notebook")

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")

In [ ]:
missing = [path for path in [INTERACTIONS_CSV, CLICKS_CSV] if not path.exists()]

if missing:
    print("Missing data files:")
    for path in missing:
        print(f"- {path}")
    print("\nPlace the CSV files in data/raw/ before running the analysis cells.")
else:
    print("Data files found. Ready to run.")


## 1. Data Preparation Helpers

These functions reproduce the original notebook's intent:

1. keep interactions before the first movie click;
2. keep only free-browsing tasks;
3. keep user-task pairs that inspected the clicked movie before clicking;
4. apply the original `Movie_Familiarity` exclusion;
5. remove consecutive duplicate fixation positions;
6. build normalized binary examination-frequency tables.

In [ ]:
USER_COL = "UserID"
TASK_COL = "TaskID"

CLICK_AOI_TYPE_COL = "Click_AOI_type"
CLICK_CAROUSEL_POSITION_COL = "Click_AOI_Carousel_position"
CLICK_MOVIE_POSITION_COL = "Click_AOI_Movie_position_in_carousel"

FIXATION_AOI_TYPE_COL = "Fixation_AOI_type"
CAROUSEL_POSITION_COL = "Fixation_AOI_Carousel_position"
MOVIE_POSITION_COL = "Fixation_AOI_Movie_position_in_carousel"
MOVIE_FAMILIARITY_COL = "Movie_Familiarity"

N_ROWS = 10
N_COLS = 15
PAGE_SIZE = 5

REQUIRED_INTERACTION_COLUMNS = {
    USER_COL,
    TASK_COL,
    CLICK_AOI_TYPE_COL,
    CLICK_CAROUSEL_POSITION_COL,
    CLICK_MOVIE_POSITION_COL,
    FIXATION_AOI_TYPE_COL,
    CAROUSEL_POSITION_COL,
    MOVIE_POSITION_COL,
}

REQUIRED_CLICK_COLUMNS = {USER_COL, TASK_COL}


def require_columns(df, required, label):
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {', '.join(missing)}")


def load_raw_data(interactions_csv, clicks_csv):
    interactions = pd.read_csv(interactions_csv)
    clicks = pd.read_csv(clicks_csv)
    require_columns(interactions, REQUIRED_INTERACTION_COLUMNS, "interactions CSV")
    require_columns(clicks, REQUIRED_CLICK_COLUMNS, "clicks CSV")
    return interactions, clicks


def interactions_before_first_movie_click(interactions):
    indexed = interactions.reset_index().rename(columns={"index": "_row_index"})
    movie_clicks = indexed[indexed[CLICK_AOI_TYPE_COL] == "Movie"].copy()

    first_click = (
        movie_clicks.groupby([USER_COL, TASK_COL])["_row_index"]
        .min()
        .rename("first_click_index")
        .reset_index()
    )

    merged = indexed.merge(first_click, on=[USER_COL, TASK_COL], how="left")
    before = merged[merged["_row_index"] <= merged["first_click_index"]].copy()
    return before.drop(columns=["first_click_index"])


def filter_free_browsing_before_click(interactions_before_click, movie_clicks, max_task_id=30):
    df = interactions_before_click.copy()
    movie_clicks = movie_clicks.copy()

    genre_mask = df[FIXATION_AOI_TYPE_COL] == "Genre"
    df.loc[genre_mask, MOVIE_POSITION_COL] = 0.0

    if max_task_id is not None:
        df = df[df[TASK_COL] <= max_task_id].copy()

    df = df[df[MOVIE_POSITION_COL].notna()].copy()

    delete_pairs = []
    click_groups = {
        key: subset
        for key, subset in movie_clicks.groupby([USER_COL, TASK_COL], sort=False)
    }

    for key, subset in df.groupby([USER_COL, TASK_COL], sort=False):
        click = click_groups.get(key)
        if click is None or click.empty:
            delete_pairs.append(key)
            continue

        first_click = click.iloc[0]
        visit = (
            subset[CAROUSEL_POSITION_COL] == first_click[CLICK_CAROUSEL_POSITION_COL]
        ) & (
            subset[MOVIE_POSITION_COL] == first_click[CLICK_MOVIE_POSITION_COL]
        )

        matches = visit.astype(int)
        shifted = matches.shift(1, fill_value=0)
        visit_start = (matches == 1) & (shifted == 0)

        if visit_start.sum() == 0:
            delete_pairs.append(key)

    if delete_pairs:
        delete_index = pd.MultiIndex.from_tuples(delete_pairs)
        row_index = pd.MultiIndex.from_frame(df[[USER_COL, TASK_COL]])
        df = df[~row_index.isin(delete_index)].copy()

    return df


def filter_by_click_familiarity(interactions, clicks, excluded_familiarities=None):
    if MOVIE_FAMILIARITY_COL not in clicks.columns:
        print("Movie_Familiarity column not found; keeping all click rows.")
        return interactions.copy()

    if excluded_familiarities is None:
        excluded_familiarities = clicks[MOVIE_FAMILIARITY_COL].unique()[-3:-1]

    keep = clicks[~clicks[MOVIE_FAMILIARITY_COL].isin(excluded_familiarities)]
    keep = keep[[TASK_COL, USER_COL]].drop_duplicates()
    return interactions.merge(keep, on=[TASK_COL, USER_COL], how="inner")


def remove_consecutive_duplicate_fixations(interactions):
    df = interactions.copy()
    duplicate = (
        (df[CAROUSEL_POSITION_COL] == df[CAROUSEL_POSITION_COL].shift(1))
        & (df[MOVIE_POSITION_COL] == df[MOVIE_POSITION_COL].shift(1))
        & (df[USER_COL] == df[USER_COL].shift(1))
        & (df[TASK_COL] == df[TASK_COL].shift(1))
    )
    return df[~duplicate].copy()


def build_binary_examination(interactions, normalize=True):
    if interactions.empty:
        return pd.DataFrame(
            columns=[
                CAROUSEL_POSITION_COL,
                MOVIE_POSITION_COL,
                "n_users_tasks",
                "exam_freq",
                "exam_row",
                "inner_freq",
            ]
        )

    stats = (
        interactions.groupby([USER_COL, TASK_COL, CAROUSEL_POSITION_COL, MOVIE_POSITION_COL])
        .size()
        .reset_index(name="n_visits")
    )

    exam_row = (
        stats[[CAROUSEL_POSITION_COL, USER_COL, TASK_COL]]
        .drop_duplicates()
        .groupby(CAROUSEL_POSITION_COL)
        .size()
        .reset_index(name="exam_row")
    )

    result = (
        stats.groupby([CAROUSEL_POSITION_COL, MOVIE_POSITION_COL])
        .size()
        .reset_index(name="n_users_tasks")
    )

    n_pairs = stats[[USER_COL, TASK_COL]].drop_duplicates().shape[0]
    result["exam_freq"] = result["n_users_tasks"] / n_pairs * 100.0
    result = result.merge(exam_row, on=CAROUSEL_POSITION_COL, how="left")
    result["inner_freq"] = result["n_users_tasks"] / result["exam_row"] * 100.0
    result = result[result[MOVIE_POSITION_COL] != 0].copy()

    if normalize and not result.empty:
        max_freq = result["exam_freq"].max()
        if max_freq > 0:
            result["exam_freq"] = result["exam_freq"] / max_freq

    return result.sort_values([CAROUSEL_POSITION_COL, MOVIE_POSITION_COL]).reset_index(drop=True)


def prepare_examination_results(interactions_csv, clicks_csv, apply_familiarity_filter=True):
    interactions, clicks = load_raw_data(interactions_csv, clicks_csv)
    movie_clicks = interactions[interactions[CLICK_AOI_TYPE_COL] == "Movie"].copy()

    before_click = interactions_before_first_movie_click(interactions)
    free_browsing = filter_free_browsing_before_click(before_click, movie_clicks)

    if apply_familiarity_filter:
        free_browsing = filter_by_click_familiarity(free_browsing, clicks)

    deduped = remove_consecutive_duplicate_fixations(free_browsing)
    user_ids = deduped[USER_COL].astype(str)

    groups = {
        "overall": deduped,
        "kinit": deduped[user_ids.str.contains("KINIT", na=False)].copy(),
        "uva": deduped[user_ids.str.contains("UvA", na=False)].copy(),
    }

    return {name: build_binary_examination(group) for name, group in groups.items()}

## 2. Build Empirical Examination Tables

In [ ]:
examination_results = prepare_examination_results(
    interactions_csv=INTERACTIONS_CSV,
    clicks_csv=CLICKS_CSV,
    apply_familiarity_filter=True,
)

for group, frame in examination_results.items():
    output_path = OUTPUT_DIR / f"examination_{group}.csv"
    frame.to_csv(output_path, index=False)
    print(f"{group:>7}: {len(frame):>3} positions -> {output_path.name}")

examination_results[TARGET_GROUP].head()

## 3. Plotting Helpers

In [ ]:
def save_or_show(fig, output_path=None, show=True):
    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, bbox_inches="tight", pad_inches=0.02)
    if show:
        plt.show()
    else:
        plt.close(fig)


def add_page_labels(ax, n_cols=N_COLS, page_size=PAGE_SIZE):
    for x in range(page_size, n_cols, page_size):
        ax.vlines(x, *ax.get_ylim(), colors="#222222", linewidth=1.8)

    n_pages = int(np.ceil(n_cols / page_size))
    for page in range(1, n_pages + 1):
        start = (page - 1) * page_size
        end = min(page * page_size, n_cols)
        center = (start + end) / 2
        ax.text(
            center / n_cols,
            1.02,
            f"Page {page}",
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
        )


def plot_examination_heatmap(examination, title, output_path=None, show=True):
    pivot = examination.pivot(
        index=CAROUSEL_POSITION_COL,
        columns=MOVIE_POSITION_COL,
        values="exam_freq",
    ).reindex(index=range(1, N_ROWS + 1), columns=range(1, N_COLS + 1))

    fig, ax = plt.subplots(figsize=(18, 6))
    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        annot=True,
        fmt=".2f",
        vmin=0,
        vmax=1,
        linewidths=0.4,
        linecolor="white",
        cbar_kws={"label": "Normalized examination frequency", "pad": 0.02},
        ax=ax,
    )

    ax.set_title(title, fontsize=14, fontweight="bold", pad=30)
    ax.set_xlabel("Movie position in carousel", fontsize=12, fontweight="bold")
    ax.set_ylabel("Carousel position", fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(N_COLS) + 0.5)
    ax.set_xticklabels(range(1, N_COLS + 1), rotation=0)
    ax.set_yticks(np.arange(N_ROWS) + 0.5)
    ax.set_yticklabels(range(1, N_ROWS + 1), rotation=0)
    add_page_labels(ax)
    fig.tight_layout()
    save_or_show(fig, output_path, show)
    return fig


def plot_discount_heatmap(discount, title, output_path=None, show=True):
    pivot = discount.pivot(
        index=CAROUSEL_POSITION_COL,
        columns=MOVIE_POSITION_COL,
        values="discount",
    ).reindex(index=range(1, N_ROWS + 1), columns=range(1, N_COLS + 1))

    fig, ax = plt.subplots(figsize=(18, 6))
    sns.heatmap(
        pivot,
        cmap="YlOrRd",
        annot=True,
        fmt=".3f",
        vmin=0,
        vmax=1,
        linewidths=0.4,
        linecolor="white",
        cbar_kws={"label": "Discount value", "pad": 0.02},
        ax=ax,
    )

    ax.set_title(title, fontsize=14, fontweight="bold", pad=30)
    ax.set_xlabel("Movie position in carousel", fontsize=12, fontweight="bold")
    ax.set_ylabel("Carousel position", fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(N_COLS) + 0.5)
    ax.set_xticklabels(range(1, N_COLS + 1), rotation=0)
    ax.set_yticks(np.arange(N_ROWS) + 0.5)
    ax.set_yticklabels(range(1, N_ROWS + 1), rotation=0)
    add_page_labels(ax)
    fig.tight_layout()
    save_or_show(fig, output_path, show)
    return fig

## 4. Empirical Examination Heatmaps

In [ ]:
for group, title in {
    "overall": "Overall Binary Examination",
    "kinit": "KINIT Binary Examination",
    "uva": "UvA Binary Examination",
}.items():
    frame = examination_results[group]
    if frame.empty:
        print(f"Skipping {group}: no rows after filtering.")
        continue

    plot_examination_heatmap(
        frame,
        title=title,
        output_path=OUTPUT_DIR / f"examination_{group}.pdf",
        show=True,
    )

## 5. Discount Function Helpers

These are the six candidate discount functions from the original notebook, rewritten as reusable notebook-local functions.

In [ ]:
def normalize_discount(matrix):
    matrix = np.asarray(matrix, dtype=float)
    max_value = np.nanmax(matrix)
    if not np.isfinite(max_value) or max_value <= 0:
        raise ValueError("Cannot normalize a discount matrix with non-positive max.")
    return matrix / max_value


def position_arrays(n_rows=N_ROWS, n_cols=N_COLS):
    rows = np.arange(1, n_rows + 1, dtype=float)[:, None]
    cols = np.arange(1, n_cols + 1, dtype=float)[None, :]
    return rows, cols


def effective_column(cols, page_size=PAGE_SIZE):
    cols = np.asarray(cols, dtype=float)
    pages = np.ceil(cols / page_size).astype(int)
    k_local = cols - (pages - 1) * page_size
    mirrored = pages * page_size - k_local + 1
    return np.where(pages == 1, k_local, mirrored).astype(float)


def swipe_counts(rows, cols, vh=5, dh=5, vv=3, dv=1):
    n_h = np.maximum(0, np.ceil((cols - vh) / dh))
    n_v = np.maximum(0, np.ceil((rows - vv) / dv))
    return n_h, n_v


def discount_to_frame(matrix):
    rows = []
    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            rows.append(
                {
                    CAROUSEL_POSITION_COL: row_idx + 1,
                    MOVIE_POSITION_COL: col_idx + 1,
                    "discount": matrix[row_idx, col_idx],
                }
            )
    return pd.DataFrame(rows)


def naive_f_pattern_discount(alpha=7, beta=6):
    rows, cols = position_arrays()
    discount = 1.0 / np.log2(alpha * rows + beta * cols)
    return normalize_discount(discount)


def naive_additive_swipe_discount(alpha=2, beta=1, gamma=9, lambda_=1):
    rows, cols = position_arrays()
    n_h, n_v = swipe_counts(rows, cols)
    denom = alpha * rows + beta * cols + gamma * n_h + lambda_ * n_v
    discount = 1.0 / np.log2(denom)
    return normalize_discount(discount)


def mirrored_f_pattern_discount(alpha=10, beta=9):
    rows, cols = position_arrays()
    eff_col = effective_column(cols)
    discount = 1.0 / np.log2(alpha * rows + beta * eff_col)
    return normalize_discount(discount)


def mirrored_additive_swipe_discount(alpha=2, beta=1, gamma=9, lambda_=1):
    rows, cols = position_arrays()
    eff_col = effective_column(cols)
    n_h, n_v = swipe_counts(rows, cols)
    denom = alpha * rows + beta * eff_col + gamma * n_h + lambda_ * n_v
    discount = 1.0 / np.log2(denom)
    return normalize_discount(discount)


def mirrored_multiplicative_swipe_discount(alpha=1, beta=9, rho_c=0.9, rho_r=0.95):
    rows, cols = position_arrays()
    eff_col = effective_column(cols)
    n_h, n_v = swipe_counts(rows, cols)
    base = 1.0 / np.log2(alpha * rows + beta * eff_col)
    discount = base * (rho_c ** n_h) * (rho_r ** n_v)
    return normalize_discount(discount)


def mirrored_row_page_discount(alpha=4, beta=9, mu=0.95, page_penalty=0.65):
    rows, cols = position_arrays()
    pages = np.ceil(cols / PAGE_SIZE).astype(int)
    eff_col = effective_column(cols)
    h_factor = np.where(pages == 1, 1.0, page_penalty)
    row_decay = mu ** (rows - 1)
    discount = (1.0 / np.log2(alpha * rows + beta * eff_col)) * h_factor * row_decay
    return normalize_discount(discount)


DISCOUNT_NAMES = {
    "naive_f_pattern": "Naive F-Pattern",
    "naive_additive_swipe": "Naive F-Pattern with Additive Swipe Penalty",
    "mirrored_f_pattern": "Mirrored F-Pattern",
    "mirrored_additive_swipe": "Mirrored F-Pattern with Additive Swipe Penalty",
    "mirrored_multiplicative_swipe": "Mirrored F-Pattern with Multiplicative Swipe Penalty",
    "mirrored_row_page": "Mirrored F-Pattern with Row-Page Discount",
}


def candidate_discount_frames():
    matrices = {
        "naive_f_pattern": naive_f_pattern_discount(),
        "naive_additive_swipe": naive_additive_swipe_discount(),
        "mirrored_f_pattern": mirrored_f_pattern_discount(),
        "mirrored_additive_swipe": mirrored_additive_swipe_discount(),
        "mirrored_multiplicative_swipe": mirrored_multiplicative_swipe_discount(),
        "mirrored_row_page": mirrored_row_page_discount(),
    }
    return {key: discount_to_frame(matrix) for key, matrix in matrices.items()}

## 6. Candidate Discount Heatmaps

In [ ]:
discounts = candidate_discount_frames()

for key, frame in discounts.items():
    frame.to_csv(OUTPUT_DIR / f"discount_{key}.csv", index=False)
    plot_discount_heatmap(
        frame,
        title=DISCOUNT_NAMES[key],
        output_path=OUTPUT_DIR / f"discount_{key}.pdf",
        show=True,
    )

## 7. Metric Scoring Helpers

Candidates are ranked by highest Spearman correlation, then highest Pearson correlation, then lowest MSE.

In [ ]:
def discount_metrics(gt, pred):
    gt = np.asarray(gt, dtype=float)
    pred = np.asarray(pred, dtype=float)

    if gt.shape != pred.shape:
        raise ValueError(f"Shape mismatch: gt={gt.shape}, pred={pred.shape}")
    if gt.size < 2:
        raise ValueError("At least two positions are required to compute metrics.")

    spearman, _ = spearmanr(gt, pred)
    pearson, _ = pearsonr(gt, pred)
    mse = float(np.mean((gt - pred) ** 2))

    return {"spearman": float(spearman), "pearson": float(pearson), "mse": mse}


def align_empirical_and_discount(examination, discount):
    merged = examination.merge(
        discount,
        on=[CAROUSEL_POSITION_COL, MOVIE_POSITION_COL],
        how="inner",
    )
    merged = merged.sort_values([CAROUSEL_POSITION_COL, MOVIE_POSITION_COL])
    if merged.empty:
        raise ValueError("No overlapping positions between examination and discount.")
    return merged.reset_index(drop=True)


def score_discount_frame(examination, discount):
    merged = align_empirical_and_discount(examination, discount)
    gt = merged["exam_freq"].to_numpy(dtype=float)
    pred = merged["discount"].to_numpy(dtype=float)

    if gt.max() > 0:
        gt = gt / gt.max()
    if pred.max() > 0:
        pred = pred / pred.max()

    metrics = discount_metrics(gt, pred)
    metrics["n_positions"] = int(len(merged))
    return metrics


def score_candidate_discounts(examination, discount_frames):
    rows = []
    for key, frame in discount_frames.items():
        rows.append(
            {
                "discount_key": key,
                "discount_name": DISCOUNT_NAMES[key],
                **score_discount_frame(examination, frame),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(["spearman", "pearson", "mse"], ascending=[False, False, True])
        .reset_index(drop=True)
    )

## 8. Candidate Scoring

In [ ]:
metric_df = score_candidate_discounts(examination_results[TARGET_GROUP], discounts)
metric_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
metric_df

## 9. Final Comparison Figure

In [ ]:
def add_position_guides(ax, n_rows=N_ROWS, n_cols=N_COLS, page_size=PAGE_SIZE):
    for row in range(n_rows):
        if row % 2 == 0:
            ax.axvspan(
                row * n_cols + 1,
                (row + 1) * n_cols + 1,
                color="#000000",
                alpha=0.04,
                zorder=0,
                lw=0,
            )

    for row in range(n_rows):
        for page_boundary in range(page_size, n_cols, page_size):
            xpos = row * n_cols + page_boundary + 0.5
            ax.axvline(
                xpos,
                color="#888888",
                linewidth=0.8,
                linestyle=":",
                alpha=0.3,
                zorder=1,
            )


def plot_candidate_comparison(examination, discount_frames, output_path=None, show=True):
    metric_df = score_candidate_discounts(examination, discount_frames)
    best_key = metric_df.loc[0, "discount_key"]
    metric_lookup = metric_df.set_index("discount_key").to_dict(orient="index")

    ordered = list(discount_frames.items())
    fig, axes = plt.subplots(2, 3, figsize=(16, 5.2), sharex=True, sharey=True)
    axes = axes.flatten()

    empirical_color = "#888888"
    candidate_color = "#2C7BB6"
    best_color = "#D7191C"
    candidate_dash = (0, (4, 2))
    best_dash = (0, (6, 2))
    letters = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    for idx, (key, discount) in enumerate(ordered):
        ax = axes[idx]
        aligned = align_empirical_and_discount(examination, discount)
        aligned = aligned.sort_values([CAROUSEL_POSITION_COL, MOVIE_POSITION_COL])

        freq = aligned["exam_freq"].to_numpy(dtype=float)
        vals = aligned["discount"].to_numpy(dtype=float)
        if freq.max() > 0:
            freq = freq / freq.max()
        if vals.max() > 0:
            vals = vals / vals.max()

        x = np.arange(1, len(freq) + 1)
        is_best = key == best_key
        metrics = metric_lookup[key]

        add_position_guides(ax)
        ax.plot(x, freq, color=empirical_color, linewidth=2.0, alpha=0.7, zorder=3)
        ax.plot(
            x,
            vals,
            color=best_color if is_best else candidate_color,
            linewidth=2.4 if is_best else 1.8,
            linestyle=best_dash if is_best else candidate_dash,
            zorder=5 if is_best else 4,
        )

        ax.set_title(f"{letters[idx]} {DISCOUNT_NAMES[key]}", pad=10)
        ax.set_ylim(0, 1.05)
        ax.set_xlim(1, len(freq))

        metric_text = (
            f"rho = {metrics['spearman']:.4f}\\n"
            f"r = {metrics['pearson']:.4f}\\n"
            f"MSE = {metrics['mse']:.4f}"
        )
        ax.text(
            0.96,
            0.94,
            metric_text,
            transform=ax.transAxes,
            ha="right",
            va="top",
            bbox={
                "boxstyle": "square,pad=0.4",
                "facecolor": "#FAFAFA",
                "edgecolor": "#DDDDDD",
                "linewidth": 0.9 if is_best else 0.5,
                "alpha": 0.88,
            },
        )

        if idx % 3 == 0:
            ax.set_ylabel("Normalized value")

    row_centers = np.array([i * N_COLS + (N_COLS + 1) / 2 for i in range(N_ROWS)])
    row_labels = [f"R{i + 1}" for i in range(N_ROWS)]
    for ax in axes[-3:]:
        ax.set_xticks(row_centers)
        ax.set_xticklabels(row_labels)
        ax.set_xlabel("Carousel position")

    legend_lines = [
        Line2D([0], [0], color=empirical_color, lw=2.0, alpha=0.7, label="Empirical examination frequency"),
        Line2D([0], [0], color=candidate_color, lw=1.8, linestyle=candidate_dash, label="Candidate discount function"),
        Line2D([0], [0], color=best_color, lw=2.4, linestyle=best_dash, label="Best-fit discount function"),
    ]
    fig.legend(
        handles=legend_lines,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=3,
        frameon=False,
        handlelength=3,
    )
    fig.subplots_adjust(left=0.04, right=0.995, bottom=0.16, top=0.93, wspace=0.08, hspace=0.22)

    save_or_show(fig, output_path, show)
    return fig, metric_df


_, metric_df = plot_candidate_comparison(
    examination_results[TARGET_GROUP],
    discounts,
    output_path=OUTPUT_DIR / "comparison_empirical_vs_candidate_discount_functions.pdf",
    show=True,
)

metric_df

## 10. Optional Parameter Search

This section keeps the spirit of the original search cells, but makes them reusable. Increase the grids if you want a broader search.

In [ ]:
def search_naive_f_pattern(examination, alpha_grid=range(1, 11), beta_grid=range(1, 11)):
    rows = examination[CAROUSEL_POSITION_COL].to_numpy(dtype=float)
    cols = examination[MOVIE_POSITION_COL].to_numpy(dtype=float)
    freq = examination["exam_freq"].to_numpy(dtype=float)
    freq = freq / freq.max()

    results = []
    for alpha in alpha_grid:
        for beta in beta_grid:
            val = alpha * rows + beta * cols
            if np.any(val <= 1):
                continue
            pred = 1 / np.log2(val)
            pred = pred / pred.max()
            rho, _ = spearmanr(freq, pred)
            r, _ = pearsonr(freq, pred)
            results.append((alpha, beta, rho, r))

    return pd.DataFrame(results, columns=["alpha", "beta", "spearman", "pearson"]).sort_values(
        ["spearman", "pearson"], ascending=[False, False]
    )


def search_mirrored_row_page(examination, alpha_grid=range(1, 11), beta_grid=range(1, 11), mu_grid=None, page_penalty_grid=None):
    if mu_grid is None:
        mu_grid = np.arange(0.05, 1.00, 0.05)
    if page_penalty_grid is None:
        page_penalty_grid = np.arange(0.05, 1.00, 0.05)

    rows = examination[CAROUSEL_POSITION_COL].to_numpy(dtype=float)
    cols = examination[MOVIE_POSITION_COL].to_numpy(dtype=float)
    freq = examination["exam_freq"].to_numpy(dtype=float)
    freq = freq / freq.max()

    pages = np.ceil(cols / PAGE_SIZE).astype(int)
    eff_col = effective_column(cols)

    results = []
    for alpha in alpha_grid:
        for beta in beta_grid:
            val = alpha * rows + beta * eff_col
            if np.any(val <= 1):
                continue
            base = 1 / np.log2(val)
            for mu in mu_grid:
                row_decay = mu ** (rows - 1)
                for page_penalty in page_penalty_grid:
                    h_factor = np.where(pages == 1, 1.0, page_penalty)
                    pred = base * h_factor * row_decay
                    pred = pred / pred.max()
                    rho, _ = spearmanr(freq, pred)
                    r, _ = pearsonr(freq, pred)
                    results.append((alpha, beta, mu, page_penalty, rho, r))

    return pd.DataFrame(
        results,
        columns=["alpha", "beta", "mu", "page_penalty", "spearman", "pearson"],
    ).sort_values(["spearman", "pearson"], ascending=[False, False])


# Example use:
# search_naive_f_pattern(examination_results[TARGET_GROUP]).head(10)
# search_mirrored_row_page(examination_results[TARGET_GROUP]).head(10)

## 11. Simulation Helpers

This is the original vs reformulated N2DCG simulation, rewritten as notebook-local code.

In [ ]:
@dataclass(frozen=True)
class DiscountParams:
    alpha: float
    beta: float
    gamma: float
    lambda_: float


@dataclass(frozen=True)
class SimulationConfig:
    n_rows: int = N_ROWS
    n_cols: int = N_COLS
    page_size: int = PAGE_SIZE
    vh: int = 5
    dh: int = 5
    vv: int = 3
    dv: int = 1
    original: DiscountParams = field(default_factory=lambda: DiscountParams(alpha=2, beta=1, gamma=9, lambda_=1))
    reformulated: DiscountParams = field(default_factory=lambda: DiscountParams(alpha=4, beta=9, gamma=0.65, lambda_=0.95))
    n_trials: int = 20_000
    p_relevant: float = 0.15
    rng_seed: int = 42
    relevance_mode: str = "binary"
    avoid_degenerate_rows: bool = True
    graded_min: int = 1
    graded_max: int = 5
    gap_thresholds: tuple = (0.0, 0.01, 0.02, 0.05, 0.10)


def examination_to_matrix(examination, n_rows=N_ROWS, n_cols=N_COLS):
    matrix = np.zeros((n_rows, n_cols), dtype=float)
    for _, row in examination.iterrows():
        row_pos = int(row[CAROUSEL_POSITION_COL]) - 1
        col_pos = int(row[MOVIE_POSITION_COL]) - 1
        if 0 <= row_pos < n_rows and 0 <= col_pos < n_cols:
            matrix[row_pos, col_pos] = float(row["exam_freq"])
    if matrix.max() > 0:
        matrix = matrix / matrix.max()
    return matrix


def compute_original_discount_for_sim(config):
    d = np.zeros((config.n_rows, config.n_cols), dtype=float)
    p = config.original
    for row in range(1, config.n_rows + 1):
        for col in range(1, config.n_cols + 1):
            n_h = max(0, int(np.ceil((col - config.vh) / config.dh)))
            n_v = max(0, int(np.ceil((row - config.vv) / config.dv)))
            denom = p.alpha * row + p.beta * col + p.gamma * n_h + p.lambda_ * n_v
            d[row - 1, col - 1] = 1.0 / np.log2(denom)
    return d / d.max()


def compute_reformulated_discount_for_sim(config):
    d = np.zeros((config.n_rows, config.n_cols), dtype=float)
    p = config.reformulated
    for row in range(1, config.n_rows + 1):
        for col in range(1, config.n_cols + 1):
            page = (col - 1) // config.page_size + 1
            k_local = col - (page - 1) * config.page_size
            eff_col = k_local if page == 1 else page * config.page_size - k_local + 1
            base = 1.0 / np.log2(p.alpha * row + p.beta * eff_col)
            page_penalty = p.gamma if page > 1 else 1.0
            row_decay = p.lambda_ ** (row - 1)
            d[row - 1, col - 1] = base * page_penalty * row_decay
    return d / d.max()


def gain(rel):
    return 2.0 ** rel - 1.0


def get_2dcg(rel, discount_matrix):
    return float(np.sum(gain(rel) * discount_matrix))


def compute_ideal_2dcg_topic_constrained(topic_to_items_rel, sorted_discount_rows):
    topic_to_items_rel = np.asarray(topic_to_items_rel, dtype=float)
    sorted_discount_rows = np.asarray(sorted_discount_rows, dtype=float)
    if topic_to_items_rel.shape[0] != sorted_discount_rows.shape[0]:
        raise ValueError("Number of topics must equal number of display rows.")

    sorted_topic_gains = np.sort(gain(topic_to_items_rel), axis=1)[:, ::-1]
    score_matrix = sorted_topic_gains @ sorted_discount_rows.T
    topic_idx, row_idx = linear_sum_assignment(-score_matrix)
    return float(score_matrix[topic_idx, row_idx].sum())


def pick(score_a, score_b):
    if score_a > score_b:
        return "A"
    if score_b > score_a:
        return "B"
    return None


def sample_num_relevant(rng, config):
    n_rel = rng.binomial(config.n_cols, config.p_relevant)
    if config.avoid_degenerate_rows:
        return int(np.clip(n_rel, 1, config.n_cols - 1))
    return int(np.clip(n_rel, 0, config.n_cols))


def generate_category_carousels(rng, config):
    category_rows = []
    for _ in range(config.n_rows):
        n_rel = sample_num_relevant(rng, config)
        if config.relevance_mode == "binary":
            row = np.array([1] * n_rel + [0] * (config.n_cols - n_rel), dtype=float)
        elif config.relevance_mode == "graded":
            grades = rng.integers(config.graded_min, config.graded_max + 1, size=n_rel)
            row = np.array(list(grades) + [0] * (config.n_cols - n_rel), dtype=float)
        else:
            raise ValueError("relevance_mode must be either 'binary' or 'graded'.")
        category_rows.append(row)
    return np.array(category_rows, dtype=float)


def sample_layout_from_category_carousels(category_rows, rng, config):
    layout = np.zeros((config.n_rows, config.n_cols), dtype=float)
    category_order = rng.permutation(config.n_rows)
    for display_row, category_id in enumerate(category_order):
        layout[display_row] = rng.permutation(category_rows[category_id])
    return layout


def sample_pair_same_candidate_sets(rng, config):
    category_rows = generate_category_carousels(rng, config)
    layout_a = sample_layout_from_category_carousels(category_rows, rng, config)
    layout_b = sample_layout_from_category_carousels(category_rows, rng, config)
    return layout_a, layout_b, category_rows


def evaluate_pair(layout_a, layout_b, category_rows, p_exam, d_orig, d_ref, sorted_p_exam, sorted_d_orig, sorted_d_ref):
    idcg_exam = compute_ideal_2dcg_topic_constrained(category_rows, sorted_p_exam)
    idcg_orig = compute_ideal_2dcg_topic_constrained(category_rows, sorted_d_orig)
    idcg_ref = compute_ideal_2dcg_topic_constrained(category_rows, sorted_d_ref)

    if idcg_exam <= 0 or idcg_orig <= 0 or idcg_ref <= 0:
        return None

    q_a = get_2dcg(layout_a, p_exam) / idcg_exam
    q_b = get_2dcg(layout_b, p_exam) / idcg_exam
    orig_a = get_2dcg(layout_a, d_orig) / idcg_orig
    orig_b = get_2dcg(layout_b, d_orig) / idcg_orig
    ref_a = get_2dcg(layout_a, d_ref) / idcg_ref
    ref_b = get_2dcg(layout_b, d_ref) / idcg_ref

    truth = pick(q_a, q_b)
    orig_pick = pick(orig_a, orig_b)
    ref_pick = pick(ref_a, ref_b)
    if truth is None or orig_pick is None or ref_pick is None:
        return None

    return {
        "Q_A": q_a,
        "Q_B": q_b,
        "gap_Q": abs(q_a - q_b),
        "orig_A": orig_a,
        "orig_B": orig_b,
        "ref_A": ref_a,
        "ref_B": ref_b,
        "truth": truth,
        "orig_pick": orig_pick,
        "ref_pick": ref_pick,
        "idcg_exam": idcg_exam,
        "idcg_orig": idcg_orig,
        "idcg_ref": idcg_ref,
    }


def run_simulation(p_exam, config):
    p_exam = np.asarray(p_exam, dtype=float)
    expected_shape = (config.n_rows, config.n_cols)
    if p_exam.shape != expected_shape:
        raise ValueError(f"p_exam must have shape {expected_shape}; got {p_exam.shape}.")

    d_orig = compute_original_discount_for_sim(config)
    d_ref = compute_reformulated_discount_for_sim(config)

    sorted_p_exam = np.sort(p_exam, axis=1)[:, ::-1]
    sorted_d_orig = np.sort(d_orig, axis=1)[:, ::-1]
    sorted_d_ref = np.sort(d_ref, axis=1)[:, ::-1]

    rng = np.random.default_rng(config.rng_seed)
    counts = Counter()
    evaluated_results = []
    both_wrong_records = []
    gap_values = []

    for trial in range(config.n_trials):
        layout_a, layout_b, category_rows = sample_pair_same_candidate_sets(rng, config)
        if np.array_equal(layout_a, layout_b):
            continue

        result = evaluate_pair(
            layout_a,
            layout_b,
            category_rows,
            p_exam,
            d_orig,
            d_ref,
            sorted_p_exam,
            sorted_d_orig,
            sorted_d_ref,
        )
        if result is None:
            continue

        truth = result["truth"]
        orig_correct = result["orig_pick"] == truth
        ref_correct = result["ref_pick"] == truth

        counts["n"] += 1
        counts["orig_correct"] += int(orig_correct)
        counts["ref_correct"] += int(ref_correct)
        counts["flip"] += int(not orig_correct and ref_correct)
        counts["orig_only"] += int(orig_correct and not ref_correct)
        counts["both_wrong"] += int(not orig_correct and not ref_correct)

        gap_values.append(result["gap_Q"])
        evaluated_results.append(result)

        if not orig_correct and not ref_correct:
            both_wrong_records.append({"trial": trial, "A": layout_a, "B": layout_b, "category_rows": category_rows, **result})

    if counts["n"] == 0:
        raise RuntimeError("No valid pairs were evaluated. Check simulation settings.")

    gap_array = np.array(gap_values, dtype=float)
    percentiles = {q: float(np.percentile(gap_array, q)) for q in [0, 10, 25, 50, 75, 90, 95, 99, 100]}

    threshold_summary = []
    for threshold in config.gap_thresholds:
        filtered = [r for r in evaluated_results if r["gap_Q"] >= threshold]
        n = len(filtered)
        if n == 0:
            threshold_summary.append({"threshold": threshold, "n": 0})
            continue
        threshold_summary.append(
            {
                "threshold": threshold,
                "n": n,
                "orig_agree": sum(r["orig_pick"] == r["truth"] for r in filtered) / n,
                "ref_agree": sum(r["ref_pick"] == r["truth"] for r in filtered) / n,
                "flip": sum(r["orig_pick"] != r["truth"] and r["ref_pick"] == r["truth"] for r in filtered) / n,
                "both_wrong": sum(r["orig_pick"] != r["truth"] and r["ref_pick"] != r["truth"] for r in filtered) / n,
            }
        )

    return {
        "config": config,
        "counts": counts,
        "n": counts["n"],
        "gap_percentiles": percentiles,
        "threshold_summary": threshold_summary,
        "both_wrong_records": both_wrong_records,
    }

In [ ]:
def format_layout(layout):
    lines = []
    for row in layout:
        tokens = ["." if value <= 0 else str(int(value)) for value in row]
        lines.append(" ".join(tokens))
    return "\\n  ".join(lines)


def format_simulation_report(result):
    counts = result["counts"]
    n = result["n"]
    config = result["config"]

    lines = [
        "=" * 80,
        "Simulation report",
        "=" * 80,
        f"Relevance mode:                         {config.relevance_mode}",
        f"Total valid pairs evaluated:            {n}",
        f"Original N2DCG agrees with truth:       {counts['orig_correct']}/{n} ({counts['orig_correct'] / n:.1%})",
        f"Reformulated N2DCG agrees with truth:   {counts['ref_correct']}/{n} ({counts['ref_correct'] / n:.1%})",
        f"Flip (orig wrong, ref correct):         {counts['flip']}/{n} ({counts['flip'] / n:.1%})",
        f"Regression (orig correct, ref wrong):   {counts['orig_only']}/{n} ({counts['orig_only'] / n:.1%})",
        f"Both wrong:                             {counts['both_wrong']}/{n} ({counts['both_wrong'] / n:.1%})",
        f"Net improvement (flip - regression):    {counts['flip'] - counts['orig_only']}/{n} ({(counts['flip'] - counts['orig_only']) / n:+.1%})",
        "",
        "=" * 80,
        "Empirical N2DCG-gap distribution",
        "=" * 80,
    ]

    for q, value in result["gap_percentiles"].items():
        lines.append(f"{q:>3}th percentile: {value:.6f}")

    lines.extend(
        [
            "",
            "=" * 80,
            "Robustness check across empirical N2DCG-gap thresholds",
            "=" * 80,
            f"{'N2DCG-gap thresh':>17} {'N':>8} {'Orig agree':>14} {'Ref agree':>14} {'Flip':>10} {'Both wrong':>14}",
            "-" * 84,
        ]
    )

    for row in result["threshold_summary"]:
        if row["n"] > 0:
            lines.append(
                f"{row['threshold']:>17.2f} {row['n']:>8} "
                f"{row['orig_agree']:>13.1%}  {row['ref_agree']:>13.1%}  "
                f"{row['flip']:>9.1%}  {row['both_wrong']:>13.1%}"
            )
        else:
            lines.append(f"{row['threshold']:>17.2f} {0:>8} {'-':>13}  {'-':>13}  {'-':>9}  {'-':>13}")

    records = sorted(result["both_wrong_records"], key=lambda item: item["gap_Q"])
    lines.extend(["", "=" * 80, "Both-wrong examples", "=" * 80])

    if not records:
        lines.append("No both-wrong examples found.")
        return "\\n".join(lines)

    def add_example(example, label):
        lines.append(f"{label} (trial {example['trial']}, empirical N2DCG-gap = {example['gap_Q']:.4f})")
        lines.append(f"  Truth:        Q_A = {example['Q_A']:.4f}, Q_B = {example['Q_B']:.4f} -> {example['truth']}")
        lines.append(f"  Original:     {example['orig_A']:.4f} vs {example['orig_B']:.4f} -> {example['orig_pick']}")
        lines.append(f"  Reformulated: {example['ref_A']:.4f} vs {example['ref_B']:.4f} -> {example['ref_pick']}")

    lines.extend(["", "5 examples with smallest empirical N2DCG-gap:", "-" * 80])
    for idx, example in enumerate(records[:5], 1):
        add_example(example, f"Example {idx}")

    lines.extend(["", "5 examples with largest empirical N2DCG-gap:", "-" * 80])
    for idx, example in enumerate(records[-5:], 1):
        add_example(example, f"Example {idx}")

    worst = records[-1]
    lines.extend(
        [
            "",
            "=" * 80,
            "Layout matrices for the largest-gap both-wrong case",
            "=" * 80,
            f"Empirical N2DCG-gap: {worst['gap_Q']:.4f} (truth picks {worst['truth']})",
            "",
            "Category-specific relevance compositions:",
            "  " + format_layout(worst["category_rows"]),
            "",
            f"Layout A (Empirical N2DCG = {worst['Q_A']:.4f}):",
            "  " + format_layout(worst["A"]),
            "",
            f"Layout B (Empirical N2DCG = {worst['Q_B']:.4f}):",
            "  " + format_layout(worst["B"]),
        ]
    )

    return "\\n".join(lines)

## 12. Binary Simulation

In [ ]:
p_exam = examination_to_matrix(examination_results[TARGET_GROUP])

binary_config = SimulationConfig(
    n_trials=N_TRIALS,
    rng_seed=RNG_SEED,
    relevance_mode="binary",
)

binary_result = run_simulation(p_exam, binary_config)
binary_report = format_simulation_report(binary_result)

(OUTPUT_DIR / "simulation_binary.txt").write_text(binary_report)
print(binary_report)

## 13. Graded Simulation

In [ ]:
graded_config = SimulationConfig(
    n_trials=N_TRIALS,
    rng_seed=RNG_SEED,
    relevance_mode="graded",
)

graded_result = run_simulation(p_exam, graded_config)
graded_report = format_simulation_report(graded_result)

(OUTPUT_DIR / "simulation_graded.txt").write_text(graded_report)
print(graded_report)

## 14. Output Files

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file() and path.name != ".gitkeep":
        print(path.relative_to(PROJECT_ROOT))